In [3]:
import collections, random, numpy as np, torch
import evaluate as eval_lib
from datasets import load_dataset
from tqdm import tqdm
from transformers import BertTokenizerFast, BertForQuestionAnswering

MODEL_DIR = "../BERT/bert_squad2_finetuned/checkpoint-84000"

tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)
model     = BertForQuestionAnswering.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# ------------------------- helpers -----------------------------

def prepare_features(examples, max_length: int = 384, doc_stride: int = 128):
    pad_right = tokenizer.padding_side == "right"
    tok = tokenizer(
        examples["question" if pad_right else "context"],
        examples["context"  if pad_right else "question"],
        truncation="longest_first", max_length=max_length, stride=doc_stride,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    mapping = tok.pop("overflow_to_sample_mapping")
    tok["example_id"] = [examples["id"][i] for i in mapping]
    return tok


def postprocess(preds, feats, examples):
    start, end = preds
    per_ex = collections.defaultdict(list)
    for i, ex_id in enumerate(feats["example_id"]):
        per_ex[ex_id].append(i)

    final, n_best, max_len = collections.OrderedDict(), 20, 30
    for ex in examples:
        cand = []
        for fi in per_ex[ex["id"]]:
            offs = feats["offset_mapping"][fi]
            for s in np.argsort(start[fi])[-n_best:]:
                for e in np.argsort(end[fi])[-n_best:]:
                    if e < s or e - s + 1 > max_len or offs[s] is None or offs[e] is None:
                        continue
                    cand.append({
                        "score": start[fi][s] + end[fi][e],
                        "start": offs[s][0],
                        "end":   offs[e][1],
                    })
        if cand:
            best = max(cand, key=lambda x: x["score"])
            final[ex["id"]] = {
                "text" : ex["context"][best["start"]: best["end"]],
                "score": best["score"],
            }
        else:
            final[ex["id"]] = {"text": "", "score": 0.0}
    return final


# ---------------------- forward pass ---------------------------
print("Loading AddSent validation set …")
examples = load_dataset("stanfordnlp/squad_adversarial", "AddSent", trust_remote_code=True)["validation"]
print(f"Tokenising {len(examples)} examples …")
features = prepare_features(examples)

batch_size = 8
all_start_logits, all_end_logits = [], []
for i in tqdm(range(0, len(features["input_ids"]), batch_size), desc="Predicting"):
    batch = {
        k: torch.tensor(v[i: i + batch_size]).to(device)
        for k, v in features.items() if k in ["input_ids", "attention_mask"]
    }
    with torch.no_grad():
        outputs = model(**batch)
    all_start_logits.extend(outputs.start_logits.cpu().numpy())
    all_end_logits.extend(outputs.end_logits.cpu().numpy())
print("Finished predictions – jump to CELL 2 for evaluation.")

Loading AddSent validation set …
Tokenising 3560 examples …


Predicting: 100%|██████████| 460/460 [05:28<00:00,  1.40it/s]

Finished predictions – jump to CELL 2 for evaluation.


In [4]:
predictions = postprocess((all_start_logits, all_end_logits), features, examples)
metric = eval_lib.load("squad")

references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
pred_list  = [{"id": ex["id"], "prediction_text": predictions[ex["id"]]["text"]} for ex in examples]

results = metric.compute(predictions=pred_list, references=references)
em_key  = "exact_match" if "exact_match" in results else "exact"
print(f"EM {results[em_key]:.2f} | F1 {results['f1']:.2f}")

# --- show a few wrong predictions ----------------------------------
wrong = [
    {"q": ex["question"],
     "p": predictions[ex["id"]]["text"],
     "g": ex["answers"]["text"][0]}
    for ex in examples
    if predictions[ex["id"]]["text"] not in ex["answers"]["text"]
]

for i, s in enumerate(random.sample(wrong, min(3, len(wrong))), 1):
    print(f"\nExample {i}\nQ: {s['q']}\nPredicted: '{s['p']}'\nGold: '{s['g']}'") 

EM 50.65 | F1 56.22

Example 1
Q: Since Denver chose white, what colors did Carolina wear in Super Bowl 50?
Predicted: ''
Gold: 'black jerseys with silver pants.'

Example 2
Q: What is the name of the stadium where Super Bowl 50 was played?
Predicted: 'Staples Center'
Gold: 'Levi's Stadium.'

Example 3
Q: How long may the Amazon rainforest be threatened, according to some computer models?
Predicted: ''
Gold: 'though the 21st century'
